# Programming Assignment: Principal Component Analysis (PCA) - Complete Guide

Welcome to the Principal Component Analysis (PCA) Programming Assignment!

Principal Component Analysis is a powerful dimensionality reduction technique that transforms high-dimensional data into a lower-dimensional space while preserving as much variance as possible. PCA is widely used for data visualization, noise reduction, feature extraction, and preprocessing before applying machine learning algorithms.

**What You Will Do in this Assignment:**

* **PCA Implementation** - Apply PCA to high-dimensional data, visualize explained variance, choose optimal components
* **PCA for Visualization** - Reduce datasets to 2D/3D for visualization, color by target, interpret clusters
* **PCA vs Feature Selection** - Compare dimensionality reduction with feature selection on model performance
* **Dimensionality Reduction Pipeline** - Build complete workflow: scale data, reduce dimensions, integrate with modeling
* **From-Scratch Implementation** - Implement PCA using covariance matrix and eigendecomposition
* **SVD Implementation** - Implement Singular Value Decomposition for dimensionality reduction

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents
- [Imports](#imports)
- [1 - PCA Implementation and Analysis](#1)
    - [1.1 - Load High-Dimensional Data](#1-1)
    - [1.2 - Apply PCA](#1-2)
        - **[Exercise 1 - apply_pca](#ex-1)**
    - [1.3 - Analyze Explained Variance](#1-3)
        - **[Exercise 2 - plot_explained_variance](#ex-2)**
    - [1.4 - Choose Optimal Components](#1-4)
        - **[Exercise 3 - select_n_components](#ex-3)**
- [2 - PCA for Visualization](#2)
    - [2.1 - Reduce to 2D](#2-1)
        - **[Exercise 4 - visualize_pca_2d](#ex-4)**
    - [2.2 - Reduce to 3D](#2-2)
        - **[Exercise 5 - visualize_pca_3d](#ex-5)**
    - [2.3 - Interpret Component Loadings](#2-3)
- [3 - PCA vs Feature Selection](#3)
    - [3.1 - Feature Selection Methods](#3-1)
        - **[Exercise 6 - select_k_best_features](#ex-6)**
    - [3.2 - Compare Model Performance](#3-2)
        - **[Exercise 7 - compare_dimensionality_reduction](#ex-7)**
- [4 - Dimensionality Reduction Pipeline](#4)
    - [4.1 - Build Complete Pipeline](#4-1)
        - **[Exercise 8 - create_pca_pipeline](#ex-8)**
    - [4.2 - Cross-Validation with Pipeline](#4-2)
- [5 - From-Scratch Implementation](#5)
    - [5.1 - PCA Using Covariance Matrix](#5-1)
        - **[Exercise 9 - pca_from_scratch](#ex-9)**
    - [5.2 - SVD-Based PCA](#5-2)
        - **[Exercise 10 - pca_using_svd](#ex-10)**
    - [5.3 - Compare Implementations](#5-3)

<a name='imports'></a>
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.datasets import make_classification, load_digits
from sklearn.metrics import accuracy_score, classification_report

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
# Import helper functions (if available)
# import helper_utils
# import unittests

<a name='1'></a>
## 1 - PCA Implementation and Analysis

PCA finds the directions (principal components) in the data that maximize variance. The first principal component captures the most variance, the second captures the second most, and so on.

Mathematically, PCA finds eigenvectors of the covariance matrix:
$$C = \frac{1}{m}X^TX$$

where $X$ is the centered data matrix.

<a name='1-1'></a>
### 1.1 - Load High-Dimensional Data

We'll work with a dataset that has many features to see how PCA helps with dimensionality reduction.

In [ ]:
# Create a synthetic high-dimensional dataset
X_synthetic, y_synthetic = make_classification(
    n_samples=500,
    n_features=50,
    n_informative=10,
    n_redundant=35,
    n_classes=3,
    n_clusters_per_class=1,
    random_state=42
)

print(f"Dataset shape: {X_synthetic.shape}")
print(f"Number of features: {X_synthetic.shape[1]}")
print(f"Number of samples: {X_synthetic.shape[0]}")
print(f"Number of classes: {len(np.unique(y_synthetic))}")
print(f"\nClass distribution: {np.bincount(y_synthetic)}")

In [ ]:
# Also load the digits dataset for visualization exercises
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"\nDigits dataset:")
print(f"  Shape: {X_digits.shape}")
print(f"  Features: {X_digits.shape[1]} (8x8 pixel images)")
print(f"  Classes: {len(np.unique(y_digits))} (digits 0-9)")

In [ ]:
# Standardize the data (critical for PCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_synthetic)
X_digits_scaled = scaler.fit_transform(X_digits)

print("Data standardized successfully!")
print(f"Mean (should be ~0): {X_scaled.mean(axis=0)[:5]}")
print(f"Std (should be ~1): {X_scaled.std(axis=0)[:5]}")

<a name='1-2'></a>
### 1.2 - Apply PCA

<a name='ex-1'></a>
**Exercise 1 - apply_pca**

Implement a function to apply PCA to the data.

**Instructions:**
- Use `sklearn.decomposition.PCA`
- Set `n_components` parameter
- Fit and transform the data
- Return the PCA object and transformed data

In [ ]:
def apply_pca(X, n_components):
    """
    Apply PCA to reduce dimensionality.
    
    Args:
        X: Feature array of shape (m, n)
        n_components: Number of principal components to keep
    
    Returns:
        pca: Fitted PCA object
        X_pca: Transformed data of shape (m, n_components)
    """
    ### START CODE HERE ### (≈ 3 lines)
    pca = None
    X_pca = None
    ### END CODE HERE ###
    
    return pca, X_pca

In [ ]:
# Test your implementation
n_components = 10
pca, X_pca = apply_pca(X_scaled, n_components)

print(f"Original shape: {X_scaled.shape}")
print(f"Reduced shape: {X_pca.shape}")
print(f"\nExplained variance ratio (first {n_components} components):")
for i, var_ratio in enumerate(pca.explained_variance_ratio_, 1):
    print(f"  PC{i}: {var_ratio:.4f} ({var_ratio*100:.2f}%)")
print(f"\nTotal explained variance: {pca.explained_variance_ratio_.sum():.4f} ({pca.explained_variance_ratio_.sum()*100:.2f}%)")

# Unit test (if available)
# unittests.test_apply_pca(apply_pca)

<a name='1-3'></a>
### 1.3 - Analyze Explained Variance

<a name='ex-2'></a>
**Exercise 2 - plot_explained_variance**

Create visualizations to understand how much variance is explained by each component.

**Instructions:**
- Plot individual explained variance for each component
- Plot cumulative explained variance
- Add a horizontal line at a threshold (e.g., 0.95)

In [ ]:
def plot_explained_variance(pca, threshold=0.95):
    """
    Plot explained variance for PCA components.
    
    Args:
        pca: Fitted PCA object
        threshold: Threshold for cumulative variance (for reference line)
    """
    ### START CODE HERE ### (≈ 20 lines)
    explained_variance = None  # Get from pca
    cumulative_variance = None  # Compute cumulative sum
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot 1: Individual explained variance
    # Use ax1.bar or ax1.plot
    
    # Plot 2: Cumulative explained variance
    # Use ax2.plot
    # Add horizontal line at threshold
    
    ### END CODE HERE ###
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Test with full PCA (all components)
pca_full, X_pca_full = apply_pca(X_scaled, X_scaled.shape[1])
plot_explained_variance(pca_full, threshold=0.95)

# Unit test (if available)
# unittests.test_plot_explained_variance(plot_explained_variance)

<a name='1-4'></a>
### 1.4 - Choose Optimal Components

<a name='ex-3'></a>
**Exercise 3 - select_n_components**

Implement a function to automatically select the number of components needed to explain a certain percentage of variance.

**Instructions:**
- Compute cumulative explained variance
- Find the index where cumulative variance exceeds the threshold
- Return the optimal number of components

In [ ]:
def select_n_components(pca, variance_threshold=0.95):
    """
    Select number of components to retain given variance threshold.
    
    Args:
        pca: Fitted PCA object with all components
        variance_threshold: Minimum cumulative variance to retain
    
    Returns:
        n_components: Number of components needed
        actual_variance: Actual cumulative variance retained
    """
    ### START CODE HERE ### (≈ 5 lines)
    cumulative_variance = None
    n_components = None  # Find first index where cumulative > threshold
    actual_variance = None
    ### END CODE HERE ###
    
    return n_components, actual_variance

In [ ]:
# Test your implementation
for threshold in [0.80, 0.90, 0.95, 0.99]:
    n_comp, actual_var = select_n_components(pca_full, threshold)
    print(f"Threshold {threshold:.0%}: Need {n_comp} components (actual variance: {actual_var:.4f})")

# Unit test (if available)
# unittests.test_select_n_components(select_n_components)

<a name='2'></a>
## 2 - PCA for Visualization

One of the most powerful applications of PCA is visualizing high-dimensional data in 2D or 3D.

<a name='2-1'></a>
### 2.1 - Reduce to 2D

<a name='ex-4'></a>
**Exercise 4 - visualize_pca_2d**

Create a 2D scatter plot of data after PCA reduction, colored by target labels.

**Instructions:**
- Apply PCA with 2 components
- Create scatter plot with PC1 on x-axis, PC2 on y-axis
- Color points by class labels
- Add explained variance to axis labels

In [ ]:
def visualize_pca_2d(X, y, title="PCA 2D Visualization"):
    """
    Visualize data in 2D using PCA.
    
    Args:
        X: Feature array
        y: Target labels
        title: Plot title
    """
    ### START CODE HERE ### (≈ 10 lines)
    # Apply PCA with 2 components
    pca, X_pca = None
    
    # Create scatter plot
    plt.figure(figsize=(10, 8))
    # Use plt.scatter with c=y for coloring
    # Add colorbar
    # Add labels with explained variance
    
    ### END CODE HERE ###
    
    plt.title(title, fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# Test with synthetic data
visualize_pca_2d(X_scaled, y_synthetic, "PCA 2D: Synthetic Data")

# Test with digits data
visualize_pca_2d(X_digits_scaled, y_digits, "PCA 2D: Handwritten Digits (0-9)")

# Unit test (if available)
# unittests.test_visualize_pca_2d(visualize_pca_2d)

<a name='2-2'></a>
### 2.2 - Reduce to 3D

<a name='ex-5'></a>
**Exercise 5 - visualize_pca_3d**

Create an interactive 3D scatter plot using the first 3 principal components.

**Instructions:**
- Apply PCA with 3 components
- Use `Axes3D` for 3D plotting
- Color points by class labels
- Label axes with explained variance

In [ ]:
def visualize_pca_3d(X, y, title="PCA 3D Visualization"):
    """
    Visualize data in 3D using PCA.
    
    Args:
        X: Feature array
        y: Target labels
        title: Plot title
    """
    ### START CODE HERE ### (≈ 12 lines)
    # Apply PCA with 3 components
    pca, X_pca = None
    
    # Create 3D plot
    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_subplot(111, projection='3d')
    
    # Create 3D scatter plot
    # Use ax.scatter with c=y for coloring
    # Add labels with explained variance
    
    ### END CODE HERE ###
    
    ax.set_title(title, fontsize=14)
    plt.show()

In [ ]:
# Test with digits data
visualize_pca_3d(X_digits_scaled, y_digits, "PCA 3D: Handwritten Digits (0-9)")

# Unit test (if available)
# unittests.test_visualize_pca_3d(visualize_pca_3d)

<a name='2-3'></a>
### 2.3 - Interpret Component Loadings

Component loadings show how much each original feature contributes to each principal component.

In [ ]:
# Analyze component loadings
pca_2d, _ = apply_pca(X_scaled, 2)
loadings = pca_2d.components_.T * np.sqrt(pca_2d.explained_variance_)

# Plot loadings for first two components
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for i in range(2):
    axes[i].bar(range(loadings.shape[0]), loadings[:, i])
    axes[i].set_xlabel('Feature Index', fontsize=12)
    axes[i].set_ylabel('Loading', fontsize=12)
    axes[i].set_title(f'PC{i+1} Loadings (Explained Variance: {pca_2d.explained_variance_ratio_[i]:.2%})', fontsize=14)
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nTop 5 features contributing to PC1:")
top_features_pc1 = np.argsort(np.abs(loadings[:, 0]))[-5:][::-1]
for idx in top_features_pc1:
    print(f"  Feature {idx}: {loadings[idx, 0]:.4f}")

<a name='3'></a>
## 3 - PCA vs Feature Selection

PCA creates new features (linear combinations), while feature selection chooses a subset of original features. Let's compare their impact on model performance.

<a name='3-1'></a>
### 3.1 - Feature Selection Methods

<a name='ex-6'></a>
**Exercise 6 - select_k_best_features**

Implement feature selection using statistical tests.

**Instructions:**
- Use `sklearn.feature_selection.SelectKBest`
- Use `f_classif` as the scoring function
- Fit and transform the data

In [ ]:
def select_k_best_features(X, y, k):
    """
    Select k best features using ANOVA F-statistic.
    
    Args:
        X: Feature array
        y: Target array
        k: Number of features to select
    
    Returns:
        selector: Fitted SelectKBest object
        X_selected: Transformed data with k features
    """
    ### START CODE HERE ### (≈ 3 lines)
    selector = None
    X_selected = None
    ### END CODE HERE ###
    
    return selector, X_selected

In [ ]:
# Test your implementation
k = 10
selector, X_selected = select_k_best_features(X_scaled, y_synthetic, k)

print(f"Original features: {X_scaled.shape[1]}")
print(f"Selected features: {X_selected.shape[1]}")
print(f"\nTop {k} feature indices:")
selected_indices = selector.get_support(indices=True)
print(selected_indices)
print(f"\nFeature scores:")
for idx in selected_indices[:5]:
    print(f"  Feature {idx}: {selector.scores_[idx]:.2f}")

# Unit test (if available)
# unittests.test_select_k_best_features(select_k_best_features)

<a name='3-2'></a>
### 3.2 - Compare Model Performance

<a name='ex-7'></a>
**Exercise 7 - compare_dimensionality_reduction**

Compare classification performance using:
1. Original features
2. PCA-reduced features
3. Selected features

**Instructions:**
- Split data into train/test
- Train logistic regression on each variant
- Compare accuracies
- Return results as dictionary

In [ ]:
def compare_dimensionality_reduction(X, y, n_components=10):
    """
    Compare classification performance with different dimensionality reduction methods.
    
    Args:
        X: Feature array
        y: Target array
        n_components: Number of components/features to use
    
    Returns:
        results: Dictionary with accuracies for each method
    """
    ### START CODE HERE ### (≈ 20 lines)
    # Split data
    X_train, X_test, y_train, y_test = None
    
    results = {}
    
    # 1. Original features
    # Train LogisticRegression and compute accuracy
    
    # 2. PCA-reduced features
    # Apply PCA to train and test
    # Train LogisticRegression and compute accuracy
    
    # 3. Selected features
    # Select features from train and test
    # Train LogisticRegression and compute accuracy
    
    ### END CODE HERE ###
    
    return results

In [ ]:
# Test your implementation
results = compare_dimensionality_reduction(X_scaled, y_synthetic, n_components=10)

print("Classification Accuracy Comparison:")
for method, accuracy in results.items():
    print(f"  {method}: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Visualize comparison
plt.figure(figsize=(10, 6))
methods = list(results.keys())
accuracies = list(results.values())
bars = plt.bar(methods, accuracies, color=['blue', 'green', 'orange'], alpha=0.7)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Dimensionality Reduction Methods Comparison', fontsize=14)
plt.ylim([0, 1.0])
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.3f}', ha='center', va='bottom', fontsize=11)
plt.grid(True, alpha=0.3, axis='y')
plt.show()

# Unit test (if available)
# unittests.test_compare_dimensionality_reduction(compare_dimensionality_reduction)

<a name='4'></a>
## 4 - Dimensionality Reduction Pipeline

In practice, we should integrate dimensionality reduction into a complete ML pipeline.

<a name='4-1'></a>
### 4.1 - Build Complete Pipeline

<a name='ex-8'></a>
**Exercise 8 - create_pca_pipeline**

Create a scikit-learn pipeline that:
1. Standardizes the data
2. Applies PCA
3. Trains a classifier

**Instructions:**
- Use `sklearn.pipeline.Pipeline`
- Add steps for scaler, PCA, and classifier
- Return the pipeline object

In [ ]:
def create_pca_pipeline(n_components=10):
    """
    Create a pipeline with scaling, PCA, and classification.
    
    Args:
        n_components: Number of PCA components
    
    Returns:
        pipeline: sklearn Pipeline object
    """
    ### START CODE HERE ### (≈ 5 lines)
    pipeline = None  # Use Pipeline with list of tuples (name, transformer)
    ### END CODE HERE ###
    
    return pipeline

In [ ]:
# Test your implementation
X_train, X_test, y_train, y_test = train_test_split(
    X_synthetic, y_synthetic, test_size=0.3, random_state=42
)

pipeline = create_pca_pipeline(n_components=15)
pipeline.fit(X_train, y_train)

# Evaluate
train_score = pipeline.score(X_train, y_train)
test_score = pipeline.score(X_test, y_test)

print(f"Pipeline with PCA (15 components):")
print(f"  Training accuracy: {train_score:.4f}")
print(f"  Test accuracy: {test_score:.4f}")

# Access PCA object from pipeline
pca_from_pipeline = pipeline.named_steps['pca']
print(f"\nExplained variance: {pca_from_pipeline.explained_variance_ratio_.sum():.4f}")

# Unit test (if available)
# unittests.test_create_pca_pipeline(create_pca_pipeline)

<a name='4-2'></a>
### 4.2 - Cross-Validation with Pipeline

Use cross-validation to find the optimal number of components.

In [ ]:
# Test different numbers of components
component_range = [5, 10, 15, 20, 25, 30]
cv_scores = []

print("Cross-validation results:")
for n_comp in component_range:
    pipeline = create_pca_pipeline(n_components=n_comp)
    scores = cross_val_score(pipeline, X_synthetic, y_synthetic, cv=5, scoring='accuracy')
    mean_score = scores.mean()
    std_score = scores.std()
    cv_scores.append(mean_score)
    print(f"  n_components={n_comp:2d}: {mean_score:.4f} (+/- {std_score:.4f})")

# Find optimal
optimal_n = component_range[np.argmax(cv_scores)]
print(f"\nOptimal number of components: {optimal_n}")

In [ ]:
# Plot cross-validation results
plt.figure(figsize=(10, 6))
plt.plot(component_range, cv_scores, marker='o', linewidth=2, markersize=8)
plt.axvline(x=optimal_n, color='red', linestyle='--', label=f'Optimal: {optimal_n} components')
plt.xlabel('Number of Components', fontsize=12)
plt.ylabel('Cross-Validation Accuracy', fontsize=12)
plt.title('Finding Optimal Number of PCA Components', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

<a name='5'></a>
## 5 - From-Scratch Implementation

Let's implement PCA from scratch to understand the mathematics!

<a name='5-1'></a>
### 5.1 - PCA Using Covariance Matrix

<a name='ex-9'></a>
**Exercise 9 - pca_from_scratch**

Implement PCA using eigendecomposition of the covariance matrix.

**Algorithm:**
1. Center the data: $X_{centered} = X - \bar{X}$
2. Compute covariance matrix: $C = \frac{1}{m}X_{centered}^T X_{centered}$
3. Compute eigenvalues and eigenvectors: $C = V\Lambda V^T$
4. Sort eigenvectors by eigenvalues (descending)
5. Select top k eigenvectors as principal components
6. Project data: $X_{pca} = X_{centered} \cdot W_k$

**Instructions:**
- Use `np.cov()` to compute covariance matrix
- Use `np.linalg.eig()` for eigendecomposition
- Sort eigenvectors by eigenvalues

In [ ]:
def pca_from_scratch(X, n_components):
    """
    Implement PCA using eigendecomposition of covariance matrix.
    
    Args:
        X: Feature array of shape (m, n)
        n_components: Number of principal components
    
    Returns:
        X_pca: Transformed data of shape (m, n_components)
        components: Principal components (eigenvectors)
        explained_variance_ratio: Proportion of variance explained
    """
    ### START CODE HERE ### (≈ 15 lines)
    # Step 1: Center the data
    X_mean = None
    X_centered = None
    
    # Step 2: Compute covariance matrix
    cov_matrix = None  # Use np.cov with rowvar=False
    
    # Step 3: Eigendecomposition
    eigenvalues, eigenvectors = None  # Use np.linalg.eig
    
    # Step 4: Sort by eigenvalues (descending)
    idx = None  # Use np.argsort
    eigenvalues = None
    eigenvectors = None
    
    # Step 5: Select top k components
    components = None
    
    # Step 6: Project data
    X_pca = None
    
    # Compute explained variance ratio
    explained_variance_ratio = None
    
    ### END CODE HERE ###
    
    return X_pca, components, explained_variance_ratio

In [ ]:
# Test your implementation
n_comp = 5
X_pca_scratch, components_scratch, var_ratio_scratch = pca_from_scratch(X_scaled, n_comp)

print(f"From-Scratch PCA Results:")
print(f"  Transformed shape: {X_pca_scratch.shape}")
print(f"  Components shape: {components_scratch.shape}")
print(f"\nExplained variance ratios:")
for i, var in enumerate(var_ratio_scratch, 1):
    print(f"  PC{i}: {var:.4f} ({var*100:.2f}%)")
print(f"\nTotal explained variance: {var_ratio_scratch.sum():.4f}")

# Unit test (if available)
# unittests.test_pca_from_scratch(pca_from_scratch)

<a name='5-2'></a>
### 5.2 - SVD-Based PCA

<a name='ex-10'></a>
**Exercise 10 - pca_using_svd**

PCA can also be implemented using Singular Value Decomposition (SVD), which is more numerically stable.

**Algorithm:**
1. Center the data
2. Compute SVD: $X_{centered} = U\Sigma V^T$
3. Principal components are columns of $V$
4. Transformed data: $X_{pca} = U\Sigma$ (or $X_{centered} \cdot V$)

**Instructions:**
- Use `np.linalg.svd()` with `full_matrices=False`
- Select top k singular vectors
- Compute explained variance from singular values

In [ ]:
def pca_using_svd(X, n_components):
    """
    Implement PCA using Singular Value Decomposition.
    
    Args:
        X: Feature array of shape (m, n)
        n_components: Number of principal components
    
    Returns:
        X_pca: Transformed data of shape (m, n_components)
        components: Principal components
        explained_variance_ratio: Proportion of variance explained
    """
    ### START CODE HERE ### (≈ 12 lines)
    m = X.shape[0]
    
    # Step 1: Center the data
    X_mean = None
    X_centered = None
    
    # Step 2: Compute SVD
    U, S, Vt = None  # Use np.linalg.svd with full_matrices=False
    
    # Step 3: Select top k components
    components = None  # V is Vt.T
    
    # Step 4: Project data
    X_pca = None  # Can use U[:, :n_components] * S[:n_components] or X_centered @ components
    
    # Compute explained variance
    explained_variance = None  # Variance = (S**2) / (m - 1)
    total_variance = None
    explained_variance_ratio = None
    
    ### END CODE HERE ###
    
    return X_pca, components, explained_variance_ratio

In [ ]:
# Test your implementation
X_pca_svd, components_svd, var_ratio_svd = pca_using_svd(X_scaled, n_comp)

print(f"SVD-Based PCA Results:")
print(f"  Transformed shape: {X_pca_svd.shape}")
print(f"  Components shape: {components_svd.shape}")
print(f"\nExplained variance ratios:")
for i, var in enumerate(var_ratio_svd, 1):
    print(f"  PC{i}: {var:.4f} ({var*100:.2f}%)")
print(f"\nTotal explained variance: {var_ratio_svd.sum():.4f}")

# Unit test (if available)
# unittests.test_pca_using_svd(pca_using_svd)

<a name='5-3'></a>
### 5.3 - Compare Implementations

Let's verify that all three implementations (sklearn, covariance, SVD) give similar results.

In [ ]:
# Compare all three implementations
n_comp = 5

# Sklearn
pca_sklearn, X_pca_sklearn = apply_pca(X_scaled, n_comp)
var_sklearn = pca_sklearn.explained_variance_ratio_

# From scratch (covariance)
X_pca_cov, _, var_cov = pca_from_scratch(X_scaled, n_comp)

# SVD
X_pca_svd, _, var_svd = pca_using_svd(X_scaled, n_comp)

# Create comparison table
comparison_df = pd.DataFrame({
    'Component': [f'PC{i+1}' for i in range(n_comp)],
    'Sklearn': var_sklearn,
    'Covariance': var_cov,
    'SVD': var_svd
})

print("Explained Variance Ratio Comparison:")
print(comparison_df.to_string(index=False))
print(f"\nTotal explained variance:")
print(f"  Sklearn:     {var_sklearn.sum():.6f}")
print(f"  Covariance:  {var_cov.sum():.6f}")
print(f"  SVD:         {var_svd.sum():.6f}")

In [ ]:
# Visualize comparison
x = np.arange(n_comp)
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width, var_sklearn, width, label='Sklearn', alpha=0.8)
ax.bar(x, var_cov, width, label='Covariance', alpha=0.8)
ax.bar(x + width, var_svd, width, label='SVD', alpha=0.8)

ax.set_xlabel('Principal Component', fontsize=12)
ax.set_ylabel('Explained Variance Ratio', fontsize=12)
ax.set_title('PCA Implementation Comparison', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([f'PC{i+1}' for i in range(n_comp)])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.show()

print("\nAll implementations produce nearly identical results!")

In [ ]:
# Compare reconstruction error
def reconstruction_error(X_original, X_reduced, components, mean):
    """Compute reconstruction error."""
    X_reconstructed = X_reduced @ components.T + mean
    return np.mean((X_original - X_reconstructed)**2)

X_mean = X_scaled.mean(axis=0)

# Note: sklearn uses slightly different centering
error_cov = reconstruction_error(X_scaled, X_pca_cov, components_scratch.T, X_mean)
error_svd = reconstruction_error(X_scaled, X_pca_svd, components_svd.T, X_mean)

print("Reconstruction Errors (MSE):")
print(f"  Covariance method: {error_cov:.6f}")
print(f"  SVD method:        {error_svd:.6f}")
print(f"\nBoth methods achieve similar reconstruction quality!")

## Summary and Next Steps

**Congratulations!** You have completed the PCA assignment. You have:

- Implemented PCA and analyzed explained variance  
- Chose optimal number of components based on variance threshold  
- Visualized high-dimensional data in 2D and 3D  
- Interpreted principal component loadings  
- Compared PCA with feature selection methods  
- Built complete dimensionality reduction pipelines  
- Implemented PCA from scratch using covariance matrix eigendecomposition  
- Implemented PCA using SVD  

**Key Takeaways:**
- PCA creates uncorrelated features that maximize variance
- Explained variance helps choose the right number of components
- PCA is excellent for visualization and noise reduction
- Always standardize data before PCA
- SVD is more numerically stable than eigendecomposition
- Pipelines integrate preprocessing and modeling elegantly

**When to use PCA:**
- High-dimensional data (many features)
- Features are correlated
- Need to visualize data
- Want to reduce noise
- Speed up training with fewer features

**When NOT to use PCA:**
- Need to interpret original features
- Features are already independent
- Small number of features
- Non-linear relationships (consider kernel PCA or autoencoders)

**Next Steps:**
- Try PCA on real datasets (images, text, genomics)
- Explore kernel PCA for non-linear dimensionality reduction
- Learn about other techniques: t-SNE, UMAP, autoencoders
- Apply PCA in deep learning pipelines!